# Normalizing Features Using MinMaxScaler

Normalization rescales numerical features into a bounded range, usually between 0 and 1. This is useful when models depend on distances, gradients, or other magnitude-sensitive calculations.

This lesson explains what MinMaxScaler does, when to use it, how it differs from StandardScaler, and how to apply it safely without data leakage.

## Why normalization matters

When numerical features live on very different scales, large-magnitude variables can dominate learning. Normalization makes each numerical column contribute within the same bounded range.

MinMaxScaler is especially useful for:

- k-Nearest Neighbors
- k-Means clustering
- Support Vector Machines
- Neural networks
- Gradient-based optimization

Tree-based models generally do not require normalization because they split on thresholds rather than distances.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder

rng = np.random.default_rng(23)
n_rows = 500

df = pd.DataFrame({
    'tenure': rng.integers(1, 72, size=n_rows),
    'MonthlyCharges': rng.normal(70, 15, size=n_rows).round(2),
    'TotalCharges': rng.normal(2500, 800, size=n_rows).round(2),
    'Contract': rng.choice(['Month-to-month', 'One year', 'Two year'], size=n_rows),
})

# Add a large outlier to show how MinMaxScaler can compress most values.
df.loc[df.index[-1], 'TotalCharges'] = 50000

logit = -0.03 * df['tenure'] + 0.016 * df['MonthlyCharges'] + 0.0003 * df['TotalCharges']
prob = 1 / (1 + np.exp(-logit))
df['Churn'] = (rng.random(n_rows) < prob).astype(int)

df.head()

## How MinMaxScaler works

The transformation is:

scaled_value = (x - x_min) / (x_max - x_min)

The smallest training value becomes 0 and the largest becomes 1. All other values are mapped proportionally between them.

In [ ]:
TARGET_COLUMN = 'Churn'
NUMERICAL_FEATURES = ['tenure', 'MonthlyCharges', 'TotalCharges']
CATEGORICAL_FEATURES = ['Contract']

X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', MinMaxScaler()),
    ]), NUMERICAL_FEATURES),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore')),
    ]), CATEGORICAL_FEATURES),
])

model = Pipeline([
    ('preprocessing', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000)),
])

model.fit(X_train, y_train)
predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

print({'test_accuracy': round(accuracy, 3)})

## Avoiding leakage while normalizing

Never fit MinMaxScaler before splitting the data. The scaler must learn minimum and maximum values from training data only. Test and production data should use the same fitted scaler through transform(), not fit_transform().

Outliers matter here: a single extreme value can stretch the range and compress most of the remaining observations into a narrow band near 0. Inspect distributions before choosing this scaler.

## Practical checklist

- Split before fitting any scaler.
- Scale only numerical features.
- Keep categorical features for encoding, not scaling.
- Save the fitted scaler with the model pipeline.
- Inspect outliers before applying MinMaxScaler.
- Use ColumnTransformer when combining scaling and encoding.

## Bonus resources

- Scikit-learn MinMaxScaler Documentation
- Comparing Different Scalers - Scikit-learn Guide